# 한국어 난독화 데이터 수집 (SFT 학습용)

- **목적**: 난독화 사이트를 Selenium으로 자동화하여 SFT 학습용 증강 데이터 수집
- **방법**: [airbnbfy.hanmesoft.com](https://airbnbfy.hanmesoft.com/) 난독화 사이트 자동화
- **사용 용도**: 대회 제공 `train.csv` 원문을 난독화 → `augmented_texts.csv` 생성
  - `augmented_texts50.csv`, `augmented_texts100.csv`는 슬라이더 범위 설정 변경 후 별도 실행
- **증강 전략**
  - 1차: `run_convert_with_range()` — 슬라이더 범위 지정해 체계적 순회
  - 2차: `run_convert_with_random()` — 랜덤 파라미터로 문장당 3회 반복
- **출력**: `input`(난독화 문장), `output`(원본 문장), `aug_type`(증강 방식) 컬럼의 CSV

In [ ]:
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import urllib.request, os, random, time
import pandas as pd
import chromedriver_autoinstaller

In [ ]:
# ── 경로 설정 ─────────────────────────────────────────────────────────────
BASE_DIR   = os.getcwd()
INPUT_CSV  = os.path.join(BASE_DIR, "data", "train.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "data", "augmented_texts.csv")  # augmented_texts50.csv / augmented_texts100.csv는 설정 변경 후 별도 실행
CKPT_CSV   = os.path.join(BASE_DIR, "data", "checkpoint.csv")

os.makedirs(os.path.join(BASE_DIR, "data"), exist_ok=True)

In [ ]:
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("lang=ko_KR")  # 한국어

# chromedriver_autoinstaller.install()

In [ ]:
train_data = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")

In [ ]:
def convert_options_param(slider1_val, slider2_val, slider3_val, slider4_val):
    """난독화 사이트의 슬라이더 4개를 JS로 직접 조작하여 파라미터를 설정한다.

    Args:
        slider1_val: 소리나는 대로 연음 강도 (0~100)
        slider2_val: 뒤에 오는 자음을 받침으로 중복 강도 (0~100)
        slider3_val: 자모를 비슷한 발음으로 변환 강도 (0~100)
        slider4_val: 의미없는 받침 추가 강도 (0~100)
    """
    for slider_id, val in zip(
        ["formRange1", "formRange2", "formRange3", "formRange4"],
        [slider1_val, slider2_val, slider3_val, slider4_val],
    ):
        slider = driver.find_element(By.ID, slider_id)
        driver.execute_script("arguments[0].value = arguments[1];", slider, val)

In [ ]:
def run_convert_with_range(range1, range2, range3, range4):
    """[1차 증강] 슬라이더 범위를 지정하면 유효한 모든 조합을 순서대로 순회한다.

    특정 강도 구간의 조합을 빠짐없이 생성할 때 사용.
    예: range1=(50, 70) → 50, 60, 70 순서로 순회
    """
    valid_values = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

    range1_values = [v for v in valid_values if range1[0] <= v <= range1[1]]
    range2_values = [v for v in valid_values if range2[0] <= v <= range2[1]]
    range3_values = [v for v in valid_values if range3[0] <= v <= range3[1]]
    range4_values = [v for v in valid_values if range4[0] <= v <= range4[1]]

    for v1 in range1_values:
        for v2 in range2_values:
            for v3 in range3_values:
                for v4 in range4_values:
                    yield (v1, v2, v3, v4)


def run_convert_with_random():
    """[2차 증강] [50, 60, 70, 80, 90, 100] 중에서 4개 파라미터를 무작위로 선택한다.

    1차 증강 이후 추가 다양성 확보를 위해 사용.
    무한 generator로 next() 호출마다 새로운 조합을 반환.
    """
    valid_values = [50, 60, 70, 80, 90, 100]

    while True:
        yield tuple(random.choice(valid_values) for _ in range(4))


gen = run_convert_with_random()

In [ ]:
next(gen)

In [ ]:
rows = []  # 리스트에 모았다가 마지막에 한 번에 DataFrame 변환 (반복 append보다 빠름)

# 크롬창 실행
driver = webdriver.Chrome(options=chrome_options)
time.sleep(1)

# 한국 난독화 사이트 접속
url = "https://airbnbfy.hanmesoft.com/"
driver.get(url)
driver.implicitly_wait(3)

# UI 요소 객체 생성
text_box        = driver.find_element(By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/textarea[1]')
convert         = driver.find_element(By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/button[1]')
convert_options = driver.find_element(By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/div/button[2]')

# 변환 옵션 패널 열기
convert_options.click()

for idx, sentence in enumerate(train_data["output"]):
    try:
        text_box.clear()
        time.sleep(1)
        text_box.send_keys(sentence)
        time.sleep(1)

        for _ in range(3):  # 문장당 3회 반복 증강
            values = next(gen)
            print(f"[{idx+1}/{len(train_data)}] Executing with values: {values}")

            convert_options_param(*values)
            time.sleep(1)

            convert.click()

            converted_sentence = driver.find_element(
                By.XPATH, '//*[@id="root"]/div[1]/div/div[3]/textarea[2]'
            )

            rows.append({
                "input":    converted_sentence.text,
                "output":   sentence,
                "aug_type": "random",  # 1차 증강은 "range", 2차 증강은 "random"
            })

            time.sleep(1)

    except Exception as e:
        print(f"[ERROR] sentence {idx+1} 스킵 → {e}")
        continue

    # 100개마다 중간 저장 (크래시 시 데이터 손실 방지)
    if (idx + 1) % 100 == 0:
        pd.DataFrame(rows).to_csv(CKPT_CSV, index=False, encoding="utf-8-sig")
        print(f"[CHECKPOINT] {idx+1}번째 문장까지 저장 완료")

text_augmentation = pd.DataFrame(rows)
text_augmentation.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"\n완료: 총 {len(text_augmentation)}개 저장 → {OUTPUT_CSV}")

In [ ]:
text_augmentation